In [5]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, r2_score

import numpy as np

import pandas as pd

import pickle

import gzip

import gc



df_aux['revenue'] = pd.to_numeric(df_aux['revenue'], errors='coerce')

df_aux['runtime'] = pd.to_numeric(df_aux['runtime'], errors='coerce')

df_aux['budget'] = pd.to_numeric(df_aux['budget'], errors='coerce')

df_aux['popularity'] = pd.to_numeric(df_aux['popularity'], errors='coerce')



df_with_runtime = df_aux[

    df_aux['runtime'].notna() &

    (df_aux['runtime'] >= 2) &

    (df_aux['runtime'] <= 250)

].copy()



df_without_runtime = df_aux[

    df_aux['runtime'].isna() |

    (df_aux['runtime'] == 0) |

    (df_aux['runtime'] < 2) |

    (df_aux['runtime'] > 250)

].copy()



print(f"Filmes com runtime válido (2 a 250 min): {len(df_with_runtime)}")

print(f"Filmes fora do escopo ou sem runtime: {len(df_without_runtime)}")



df_with_runtime['release_year'] = pd.to_datetime(df_with_runtime['release_date'], errors='coerce').dt.year



df_with_runtime['keywords'] = df_with_runtime['keywords'].astype(str).fillna('')

df_with_runtime['is_short_keyword'] = df_with_runtime['keywords'].str.contains('short', case=False, regex=False).astype('int8')



numerical_features = ['vote_average', 'vote_count', 'release_year', 'budget', 'popularity', 'is_short_keyword']

multi_value_categorical_features = ['genres', 'production_companies', 'production_countries']

single_value_categorical_features = []



all_base_features = numerical_features + multi_value_categorical_features + single_value_categorical_features



df_train = df_with_runtime[all_base_features + ['runtime']].copy()

df_train.dropna(subset=['vote_average', 'vote_count', 'release_year', 'budget', 'popularity'], inplace=True)



print(f"\nFilmes válidos para treinamento: {len(df_train)}")



top_n_categories = 30

cols_to_drop = []



for col in multi_value_categorical_features:

    df_train[col] = df_train[col].astype(str).fillna('')

   

    item_counts = df_train[col].str.split(', ').explode().str.strip().value_counts()

    top_items = item_counts[item_counts.index != ''].head(top_n_categories).index.tolist()

   

    for item_name in top_items:

        df_train[f'{col}_{item_name}'] = df_train[col].str.contains(item_name, regex=False, na=False).astype('int8')

   

    cols_to_drop.append(col)



df_train.drop(columns=cols_to_drop, inplace=True)



df_train = pd.get_dummies(df_train, columns=[col for col in single_value_categorical_features], drop_first=True)



print(f"\nFilmes válidos para treinamento após engenharia de features: {len(df_train)}")

print(f"Número total de features criadas: {len(df_train.columns) - 1}")



X = df_train.drop('runtime', axis=1)

y = np.log1p(df_train['runtime'])



features_for_prediction_final = X.columns.tolist()



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



del df_with_runtime

del df_without_runtime

del df_train

del X

del y

gc.collect()



model = RandomForestRegressor(

    n_estimators=100,

    max_depth=15,

    min_samples_leaf=8,

    random_state=42,

    n_jobs=-1

)

model.fit(X_train, y_train)



y_pred_log = model.predict(X_test)



y_test_original = np.expm1(y_test)

y_pred_original = np.expm1(y_pred_log)



r2 = r2_score(y_test_original, y_pred_original)

rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))



print(f"\n--- Avaliação do Modelo para Previsão de Runtime ---")

print(f"R² Score: {r2:.4f}")

print(f"RMSE: {rmse:,.2f}")



print("\n--- Top 20 Importância das Features ---")

feature_importances = sorted(zip(features_for_prediction_final, model.feature_importances_), key=lambda x: x[1], reverse=True)

for feat, imp in feature_importances[:20]:

    print(f"{feat}: {imp:.4f}")



with gzip.open('models/random_forest_runtime_model.pkl.gz', 'wb') as f:

    pickle.dump(model, f)



with open('models/features_runtime_model.pkl', 'wb') as f:

    pickle.dump(features_for_prediction_final, f)

Filmes com runtime válido (2 a 250 min): 668981
Filmes fora do escopo ou sem runtime: 282574

Filmes válidos para treinamento: 596880

Filmes válidos para treinamento após engenharia de features: 596880
Número total de features criadas: 86

--- Avaliação do Modelo para Previsão de Runtime ---
R² Score: 0.3794
RMSE: 34.41

--- Top 20 Importância das Features ---
popularity: 0.3672
release_year: 0.1690
genres_Animation: 0.1619
is_short_keyword: 0.0704
genres_None: 0.0383
genres_Documentary: 0.0253
production_countries_India: 0.0181
vote_count: 0.0169
production_countries_Japan: 0.0157
production_companies_None: 0.0148
budget: 0.0135
vote_average: 0.0134
production_countries_United States of America: 0.0114
genres_Music: 0.0077
genres_Drama: 0.0077
genres_Family: 0.0066
production_countries_Mexico: 0.0048
production_countries_None: 0.0048
production_countries_China: 0.0048
production_countries_Canada: 0.0032
